# RAG Databricks Bluetab - Embedding Serving Endpoint Creation

## Overview
This notebook creates a serving endpoint for the registered embedding model, enabling real-time inference for text embeddings.

## Features
- Automated endpoint creation with error handling
- Configurable workload sizes and scaling options
- Health checking and status monitoring
- Integration with MLflow model registry
- Parameterized deployment for different environments

## Endpoint Configuration
- **Workload Size**: Configurable (Small, Medium, Large)
- **Auto-scaling**: Enabled with scale-to-zero for cost optimization
- **Traffic Management**: 100% traffic to latest model version
- **Monitoring**: Built-in logging and metrics

## Dependencies
- Run `00 Configuration and Utils` notebook first
- Ensure embedding model is registered (run `03 Register embedding model`)

In [ ]:
# Load shared configuration
%run "./00 Configuration and Utils"

# Iniciar child run para esta tarea
start_child_run("04_create_embedding_endpoint")

In [ ]:
# Create endpoint-specific widgets
dbutils.widgets.dropdown("workload_size", "Small", ["Small", "Medium", "Large"], "Workload Size")
dbutils.widgets.dropdown("scale_to_zero", "true", ["true", "false"], "Enable Scale to Zero")
dbutils.widgets.text("model_version", "latest", "Model Version (or 'latest')")
dbutils.widgets.text("endpoint_suffix", "", "Endpoint Name Suffix (optional)")

# Get widget values
WORKLOAD_SIZE = dbutils.widgets.get("workload_size")
SCALE_TO_ZERO = dbutils.widgets.get("scale_to_zero").lower() == "true"
MODEL_VERSION = dbutils.widgets.get("model_version")
ENDPOINT_SUFFIX = dbutils.widgets.get("endpoint_suffix")

# Build endpoint name with optional suffix
if ENDPOINT_SUFFIX:
    EMBEDDING_ENDPOINT_NAME = f"{EMBEDDING_ENDPOINT}_{ENDPOINT_SUFFIX}"
else:
    EMBEDDING_ENDPOINT_NAME = EMBEDDING_ENDPOINT

print(f"Endpoint Configuration:")
print(f"  Name: {EMBEDDING_ENDPOINT_NAME}")
print(f"  Model: {EMBEDDING_MODEL_FULL}")
print(f"  Version: {MODEL_VERSION}")
print(f"  Workload Size: {WORKLOAD_SIZE}")
print(f"  Scale to Zero: {SCALE_TO_ZERO}")

In [ ]:
import mlflow
from mlflow.deployments import get_deploy_client

# Start MLflow run for endpoint creation
with mlflow.start_run(run_name=f"04_Create_Embedding_Endpoint_{ENVIRONMENT}") as run:
    mlflow.log_param("step", "embedding_endpoint_creation")
    mlflow.log_param("environment", ENVIRONMENT)
    mlflow.log_param("endpoint_name", EMBEDDING_ENDPOINT_NAME)
    mlflow.log_param("model_name", EMBEDDING_MODEL_FULL)
    mlflow.log_param("model_version", MODEL_VERSION)
    mlflow.log_param("workload_size", WORKLOAD_SIZE)
    mlflow.log_param("scale_to_zero", SCALE_TO_ZERO)
    
    log_step("endpoint_creation", "started", f"Creating endpoint {EMBEDDING_ENDPOINT_NAME}")

    # Initialize deployment client
    client = get_deploy_client("databricks")
    
    log_step("client_initialization", "success", "Deployment client initialized")

In [ ]:
# Check if endpoint already exists
log_step("endpoint_check", "started", f"Checking if endpoint {EMBEDDING_ENDPOINT_NAME} exists")

try:
    # List existing endpoints
    existing_endpoints = client.list_endpoints()
    endpoint_names = [ep.get('name', '') for ep in existing_endpoints.get('endpoints', [])]
    
    endpoint_exists = EMBEDDING_ENDPOINT_NAME in endpoint_names
    
    if endpoint_exists:
        log_step("endpoint_check", "found", f"Endpoint {EMBEDDING_ENDPOINT_NAME} already exists")
        print(f"⚠️  Endpoint '{EMBEDDING_ENDPOINT_NAME}' already exists!")
        
        # Get endpoint details
        try:
            endpoint_details = client.get_endpoint(EMBEDDING_ENDPOINT_NAME)
            current_state = endpoint_details.get('state', {}).get('ready', 'unknown')
            
            print(f"📊 Current endpoint status: {current_state}")
            mlflow.log_param("endpoint_exists", True)
            mlflow.log_param("current_state", current_state)
            
        except Exception as e:
            log_step("endpoint_details", "failed", f"Could not get endpoint details: {e}")
    else:
        log_step("endpoint_check", "not_found", f"Endpoint {EMBEDDING_ENDPOINT_NAME} does not exist - will create")
        mlflow.log_param("endpoint_exists", False)
        
except Exception as e:
    log_step("endpoint_check", "error", f"Error checking endpoints: {e}")
    endpoint_exists = False
    mlflow.log_param("endpoint_check_error", str(e))

In [ ]:
# Create the endpoint if it doesn't exist
if not endpoint_exists:
    log_step("endpoint_creation", "started", f"Creating new endpoint {EMBEDDING_ENDPOINT_NAME}")
    
    try:
        # Resolve model version if 'latest' is specified
        if MODEL_VERSION.lower() == "latest":
            model_versions = client.search_model_versions(f"name='{EMBEDDING_MODEL_FULL}'")
            if model_versions:
                # Get the latest version
                latest_version = max([int(mv.version) for mv in model_versions])
                resolved_version = str(latest_version)
            else:
                resolved_version = "1"  # Fallback to version 1
        else:
            resolved_version = MODEL_VERSION
            
        mlflow.log_param("resolved_model_version", resolved_version)
        
        # Define endpoint configuration
        endpoint_config = {
            "served_entities": [
                {
                    "name": f"{EMBEDDING_ENDPOINT_NAME}-entity",
                    "entity_name": EMBEDDING_MODEL_FULL,
                    "entity_version": resolved_version,
                    "workload_size": WORKLOAD_SIZE,
                    "scale_to_zero_enabled": SCALE_TO_ZERO
                }
            ],
            "traffic_config": {
                "routes": [
                    {
                        "served_model_name": f"{EMBEDDING_ENDPOINT_NAME}-entity",
                        "traffic_percentage": 100
                    }
                ]
            }
        }
        
        # Log configuration details
        mlflow.log_dict(endpoint_config, "endpoint_config.json")
        
        print(f"📝 Endpoint Configuration:")
        print(f"   Model: {EMBEDDING_MODEL_FULL} (v{resolved_version})")
        print(f"   Workload: {WORKLOAD_SIZE}")
        print(f"   Scale to Zero: {SCALE_TO_ZERO}")
        
        # Create the endpoint
        log_step("endpoint_deployment", "started", "Deploying endpoint...")
        
        endpoint = client.create_endpoint(
            name=EMBEDDING_ENDPOINT_NAME,
            config=endpoint_config
        )
        
        mlflow.log_param("endpoint_creation_status", "success")
        mlflow.log_param("endpoint_uri", endpoint.get('name', ''))
        
        log_step("endpoint_creation", "success", f"Endpoint {EMBEDDING_ENDPOINT_NAME} created successfully")
        
        print(f"✅ Endpoint '{EMBEDDING_ENDPOINT_NAME}' created successfully!")
        print(f"🚀 Deployment initiated - endpoint will be available shortly")
        
    except Exception as e:
        mlflow.log_param("endpoint_creation_status", "failed")
        mlflow.log_param("endpoint_creation_error", str(e))
        
        log_step("endpoint_creation", "failed", f"Error creating endpoint: {e}")
        print(f"❌ Error creating endpoint: {e}")
        
        # Try to provide helpful error information
        if "already exists" in str(e).lower():
            print("💡 The endpoint might have been created between checks. Try running the check again.")
        elif "quota" in str(e).lower():
            print("💡 You might have reached your endpoint quota. Check your workspace limits.")
        elif "permission" in str(e).lower():
            print("💡 Check that you have permission to create serving endpoints.")
            
        raise e
else:
    log_step("endpoint_creation", "skipped", "Endpoint already exists")
    print(f"✅ Endpoint '{EMBEDDING_ENDPOINT_NAME}' already exists - no action needed")

In [ ]:
# Wait for endpoint to be ready and monitor status
import time

log_step("endpoint_monitoring", "started", "Monitoring endpoint deployment status")

max_wait_time = 600  # 10 minutes max wait
check_interval = 30  # Check every 30 seconds
elapsed_time = 0

print(f"⏳ Waiting for endpoint to be ready (max {max_wait_time//60} minutes)...")

try:
    while elapsed_time < max_wait_time:
        try:
            endpoint_status = client.get_endpoint(EMBEDDING_ENDPOINT_NAME)
            
            # Extract status information
            state = endpoint_status.get('state', {})
            ready_status = state.get('ready', 'unknown')
            config_update = state.get('config_update', 'unknown')
            
            print(f"⏱️  Time: {elapsed_time//60}m {elapsed_time%60}s - Status: {ready_status} - Config: {config_update}")
            
            # Log status to MLflow
            mlflow.log_metric(f"status_check_{elapsed_time}", 1 if ready_status == 'READY' else 0)
            
            if ready_status == 'READY':
                log_step("endpoint_monitoring", "success", f"Endpoint is ready after {elapsed_time} seconds")
                mlflow.log_metric("deployment_time_seconds", elapsed_time)
                print(f"✅ Endpoint '{EMBEDDING_ENDPOINT_NAME}' is now READY!")
                break
            elif ready_status == 'FAILED':
                log_step("endpoint_monitoring", "failed", "Endpoint deployment failed")
                mlflow.log_param("deployment_status", "failed")
                print(f"❌ Endpoint deployment failed!")
                break
                
        except Exception as e:
            print(f"⚠️  Error checking status: {e}")
            
        time.sleep(check_interval)
        elapsed_time += check_interval
    else:
        # Timeout reached
        log_step("endpoint_monitoring", "timeout", f"Timeout after {max_wait_time} seconds")
        mlflow.log_param("deployment_status", "timeout")
        print(f"⏰ Timeout: Endpoint not ready after {max_wait_time//60} minutes")
        
except Exception as e:
    log_step("endpoint_monitoring", "error", f"Error monitoring endpoint: {e}")
    print(f"❌ Error monitoring endpoint: {e}")

In [ ]:
# Test the endpoint with a sample request
log_step("endpoint_testing", "started", "Testing endpoint with sample data")

try:
    # Prepare test data
    test_data = {
        "dataframe_split": {
            "columns": ["input"],
            "data": [
                ["This is a test sentence for the embedding model"],
                ["Another test sentence to verify the endpoint works"]
            ]
        }
    }
    
    print("🧪 Testing endpoint with sample data...")
    
    # Make prediction request
    response = client.predict(
        endpoint=EMBEDDING_ENDPOINT_NAME,
        inputs=test_data
    )
    
    # Validate response
    if "predictions" in response:
        predictions = response["predictions"]
        
        # Check response format
        if isinstance(predictions, list) and len(predictions) > 0:
            embedding_dim = len(predictions[0]) if isinstance(predictions[0], list) else 0
            
            mlflow.log_metric("test_predictions_count", len(predictions))
            mlflow.log_metric("test_embedding_dimension", embedding_dim)
            mlflow.log_param("endpoint_test_status", "success")
            
            log_step("endpoint_testing", "success", f"Test successful - {len(predictions)} embeddings of {embedding_dim}D")
            
            print(f"✅ Endpoint test successful!")
            print(f"   📊 Generated {len(predictions)} embeddings")
            print(f"   📏 Embedding dimension: {embedding_dim}")
            print(f"   🔢 Sample embedding (first 5 values): {predictions[0][:5] if predictions[0] else 'N/A'}")
            
        else:
            log_step("endpoint_testing", "failed", "Invalid response format")
            print(f"❌ Invalid response format: {predictions}")
    else:
        log_step("endpoint_testing", "failed", "No predictions in response")
        print(f"❌ No predictions in response: {response}")
        
except Exception as e:
    mlflow.log_param("endpoint_test_status", "failed")
    mlflow.log_param("endpoint_test_error", str(e))
    
    log_step("endpoint_testing", "failed", f"Endpoint test failed: {e}")
    print(f"❌ Endpoint test failed: {e}")
    
    # Provide troubleshooting hints
    if "not found" in str(e).lower():
        print("💡 Endpoint might not be fully deployed yet. Wait a few more minutes and try again.")
    elif "timeout" in str(e).lower():
        print("💡 Request timed out. The endpoint might be starting up - try again in a moment.")

In [ ]:
# Provide final summary and next steps
log_step("endpoint_creation", "completed", "Endpoint creation process finished")

print("="*60)
print("EMBEDDING ENDPOINT CREATION SUMMARY")
print("="*60)
print(f"Endpoint Name: {EMBEDDING_ENDPOINT_NAME}")
print(f"Model: {EMBEDDING_MODEL_FULL}")
print(f"Environment: {ENVIRONMENT}")
print(f"Workload Size: {WORKLOAD_SIZE}")
print(f"Scale to Zero: {SCALE_TO_ZERO}")
print()

# Get final endpoint status
try:
    final_status = client.get_endpoint(EMBEDDING_ENDPOINT_NAME)
    ready_state = final_status.get('state', {}).get('ready', 'unknown')
    print(f"Final Status: {ready_state}")
    
    if ready_state == 'READY':
        print("✅ Endpoint is ready for use!")
        print()
        print("🔗 Endpoint URL:")
        endpoint_url = build_endpoint_url(EMBEDDING_ENDPOINT_NAME)
        print(f"   {endpoint_url}")
        
        mlflow.log_param("endpoint_url", endpoint_url)
        mlflow.log_param("final_status", "ready")
        
    else:
        print(f"⚠️  Endpoint status: {ready_state}")
        print("   Check the Databricks Serving UI for more details")
        mlflow.log_param("final_status", ready_state)
        
except Exception as e:
    print(f"❌ Could not get final status: {e}")
    mlflow.log_param("final_status", "error")

print()
print("📋 Next Steps:")
print("   1. Verify endpoint is working in Databricks Serving UI")
print("   2. Update the endpoint URL in your configuration if needed")
print("   3. Proceed to create embeddings using this endpoint")
print("="*60)

# Finalizar child run
try:
    end_child_run("success")
    print("✅ Child run finalizada correctamente")
except Exception as e:
    print(f"⚠️ Error finalizando child run: {e}")
    end_child_run("failed")